In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\Air_Quality_Data\AQI_daily_2024_DTU_Delhi_CPCB_2024.xlsx")

In [3]:
df

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,244.0,145.0,192.0,120.0,191.0,273.0,85.0,55.0,68.0,136.0,275.0,285.0
1,2,259.0,131.0,111.0,145.0,191.0,180.0,128.0,72.0,73.0,162.0,258.0,220.0
2,3,246.0,152.0,126.0,NaN,301.0,162.0,122.0,62.0,98.0,156.0,275.0,274.0
3,4,324.0,209.0,140.0,159.0,281.0,214.0,60.0,60.0,66.0,174.0,326.0,166.0
4,5,246.0,136.0,119.0,161.0,294.0,287.0,88.0,46.0,66.0,123.0,337.0,155.0
5,6,220.0,112.0,121.0,167.0,248.0,210.0,64.0,50.0,97.0,136.0,355.0,184.0
6,7,308.0,166.0,143.0,229.0,310.0,214.0,76.0,47.0,70.0,109.0,394.0,206.0
7,8,314.0,NaN,149.0,NaN,248.0,232.0,58.0,54.0,68.0,150.0,386.0,302.0
8,9,254.0,155.0,124.0,203.0,180.0,146.0,105.0,52.0,88.0,71.0,319.0,152.0
9,10,168.0,253.0,143.0,242.0,210.0,127.0,140.0,58.0,91.0,116.0,334.0,163.0


In [4]:
df_cleaned = df.drop_duplicates(keep='first')
df_cleaned.shape

(41, 13)

In [5]:
# Replace 'NA', empty strings, and explicit NaNs with np.nan for consistency
df_cleaned = df_cleaned.replace('NA', np.nan)
df_cleaned = df_cleaned.replace(r'^\s*$', np.nan, regex=True)

# Convert all columns except 'Day' to numeric
for col in df_cleaned.columns:
    if col != 'Day':
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

# Fill missing values with the mean of each column
df_filled = df_cleaned.fillna(df_cleaned.mean(numeric_only=True))

In [6]:
def handle_outliers_iqr(df):
    for col in df.columns:
        if col != 'Day' and df[col].dtype != 'O':
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mean_val = df[col].mean()
            df[col] = np.where((df[col] < lower) | (df[col] > upper), mean_val, df[col])
    return df

df_no_outliers = handle_outliers_iqr(df_filled.copy())

In [7]:
# Drop non-feature rows if present, and reset index
df_ml_ready = df_no_outliers.copy()

# If your dataset contains summary/statistical rows (not data for 'Day'), remove them
df_ml_ready = df_ml_ready[df_ml_ready['Day'].apply(lambda x: str(x).isdigit())]
df_ml_ready = df_ml_ready.reset_index(drop=True)

# Optional: Convert 'Day' to int if needed
df_ml_ready['Day'] = df_ml_ready['Day'].astype(int)

df_ml_ready

,Day,January,February,March,April,May,June,July,August,September,October,November,December
0,1,244.0,145.000000,192.0,120.0,191.0,273.000000,85.000000,55.000000,68.000000,136.000000,275.000000,285.0
1,2,259.0,131.000000,111.0,145.0,191.0,180.000000,128.000000,72.000000,73.000000,162.000000,258.000000,220.0
2,3,246.0,152.000000,126.0,170.5,301.0,162.000000,122.000000,62.000000,98.000000,156.000000,275.000000,274.0
3,4,324.0,209.000000,140.0,159.0,281.0,214.000000,60.000000,60.000000,66.000000,174.000000,326.000000,166.0
4,5,246.0,136.000000,119.0,161.0,294.0,161.941176,88.000000,46.000000,66.000000,123.000000,337.000000,155.0
5,6,220.0,112.000000,121.0,167.0,248.0,210.000000,64.000000,50.000000,97.000000,136.000000,355.000000,184.0
6,7,308.0,166.000000,143.0,229.0,310.0,214.000000,76.000000,47.000000,70.000000,109.000000,394.000000,206.0
7,8,314.0,168.516129,149.0,170.5,248.0,232.000000,58.000000,54.000000,68.000000,150.000000,386.000000,302.0
8,9,254.0,155.000000,124.0,203.0,180.0,146.000000,105.000000,52.000000,88.000000,71.000000,319.000000,152.0
9,10,168.0,253.000000,143.0,242.0,210.0,127.000000,82.205882,58.000000,91.000000,116.000000,334.000000,163.0
